# Sada Reviews — CAMeLBERT Fine-tuning

Colab notebook to fine-tune CAMeLBERT for app review theme classification using the project sample: seven themes and seven language/dialect slices.

**Methodological note:** CAMeLBERT is an Arabic model. English and Arabizi are measured separately, and equivalent performance is not assumed. The current dataset is synthetic; results are for educational experiments only.

In [ ]:
!pip -q install transformers datasets accelerate scikit-learn pandas matplotlib


## 1. Upload the project data

Download data/sample_reviews.csv from this repository and upload it to Colab. Columns include review, category, sentiment, language, dialect, and split.

In [ ]:
from google.colab import files
uploaded = files.upload()
csv_name = next(name for name in uploaded if name.lower().endswith(".csv"))


In [ ]:
import inspect
import json
import re
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

df = pd.read_csv(csv_name)
required = {"review", "category", "language", "dialect", "split"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

def normalize_review(text):
    text = str(text).strip()
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}", " [EMAIL] ", text)
    text = re.sub(r"(?<!\d)(?:05\d{8}|\+9665\d{8})(?!\d)", " [PHONE] ", text)
    return re.sub(r"\s+", " ", text).strip()

df["review"] = df["review"].map(normalize_review)
train_df = df[df["split"] == "train"].copy().reset_index(drop=True)
test_df = df[df["split"] == "test"].copy().reset_index(drop=True)
labels = sorted(df["category"].unique())
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}
train_df["label"] = train_df["category"].map(label2id)
test_df["label"] = test_df["category"].map(label2id)
print("Rows:", len(df), "| Train:", len(train_df), "| Test:", len(test_df))
print("Labels:", label2id)
print("Train label counts:\n", train_df["category"].value_counts())

## 2. Load CAMeLBERT and tokenize

Use CAMeLBERT-Mix to classify review themes. Dialect-specific evaluation is especially important for English and Arabizi.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding, EarlyStoppingCallback

model_name = "CAMeL-Lab/bert-base-arabic-camelbert-mix"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=len(labels), label2id=label2id, id2label=id2label
)

train_hf = Dataset.from_pandas(train_df[["review", "label"]], preserve_index=False)
test_hf = Dataset.from_pandas(test_df[["review", "label"]], preserve_index=False)
def tokenize_batch(batch):
    return tokenizer(batch["review"], truncation=True, max_length=128)
train_hf = train_hf.map(tokenize_batch, batched=True, remove_columns=["review"])
test_hf = test_hf.map(tokenize_batch, batched=True, remove_columns=["review"])
train_hf = train_hf.rename_column("label", "labels")
test_hf = test_hf.rename_column("label", "labels")
collator = DataCollatorWithPadding(tokenizer=tokenizer)
print("Tokenized train/test:", len(train_hf), len(test_hf))

## 3. Fine-tune the model

Start with three epochs and select the best checkpoint by Macro-F1. The held-out set contains generated paraphrases, not real store reviews.

In [ ]:
def compute_metrics(eval_pred):
    logits, gold = eval_pred
    pred = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(gold, pred),
        "macro_f1": f1_score(gold, pred, average="macro", zero_division=0),
    }

arg_names = inspect.signature(TrainingArguments).parameters
eval_key = "eval_strategy" if "eval_strategy" in arg_names else "evaluation_strategy"
training_kwargs = {
    "output_dir": "./sada_camelbert_checkpoints",
    "num_train_epochs": 3,
    "per_device_train_batch_size": 16,
    "per_device_eval_batch_size": 32,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "save_strategy": "epoch",
    "load_best_model_at_end": True,
    "metric_for_best_model": "macro_f1",
    "greater_is_better": True,
    "report_to": "none",
    "fp16": torch.cuda.is_available(),
}
training_kwargs[eval_key] = "epoch"
args = TrainingArguments(**training_kwargs)
trainer_kwargs = {
    "model": model, "args": args,
    "train_dataset": train_hf, "eval_dataset": test_hf,
    "data_collator": collator,
    "compute_metrics": compute_metrics,
    "callbacks": [EarlyStoppingCallback(early_stopping_patience=2)],
}
if "processing_class" in inspect.signature(Trainer.__init__).parameters:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer
trainer = Trainer(**trainer_kwargs)
trainer.train()

## 4. Evaluation and dialect analysis

Report Macro-F1, Accuracy, per-class scores, and dialect/language slices.

In [ ]:
prediction_output = trainer.predict(test_hf)
y_pred = np.argmax(prediction_output.predictions, axis=1)
print("CAMeLBERT Accuracy:", round(accuracy_score(test_df["label"], y_pred), 4))
print("CAMeLBERT Macro-F1:", round(f1_score(test_df["label"], y_pred, average="macro", zero_division=0), 4))
print(classification_report(test_df["label"], y_pred, target_names=labels, zero_division=0))
test_df["predicted"] = [id2label[int(i)] for i in y_pred]
test_df["correct"] = test_df["category"] == test_df["predicted"]
dialect_rows = []
for (dialect, language), part in test_df.groupby(["dialect", "language"], dropna=False):
    dialect_rows.append({
        "dialect": dialect, "language": language, "n": len(part),
        "accuracy": accuracy_score(part["category"], part["predicted"]),
        "macro_f1": f1_score(part["category"], part["predicted"], average="macro", labels=labels, zero_division=0),
    })
dialect_report = pd.DataFrame(dialect_rows)
display(dialect_report.sort_values("macro_f1"))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
ConfusionMatrixDisplay.from_predictions(
    test_df["category"], test_df["predicted"], labels=labels,
    display_labels=labels, xticks_rotation=45, cmap="Blues", ax=axes[0]
)
axes[0].set_title("CAMeLBERT — Confusion Matrix")
dialect_report.sort_values("macro_f1").plot(
    kind="barh", x="dialect", y="macro_f1", legend=False,
    color="#2878a5", ax=axes[1]
)
axes[1].set_xlim(0, 1)
axes[1].set_title("Macro-F1 by dialect / language")
axes[1].set_xlabel("Macro-F1")
plt.tight_layout()
plt.savefig("sada_camelbert_evaluation.png", dpi=180, bbox_inches="tight")
plt.show()

## 5. Save the model and outputs

Save the fine-tuned weights for the demo. Do not upload real reviews containing personal information.

In [ ]:
import os
output_dir = "./models/sada_camelbert_finetuned"
os.makedirs("./models", exist_ok=True)
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
test_df.to_csv("sada_camelbert_test_predictions.csv", index=False)
!zip -qr sada_camelbert_finetuned.zip models/sada_camelbert_finetuned
print("Saved:", output_dir)
files.download("sada_camelbert_evaluation.png")
files.download("sada_camelbert_test_predictions.csv")
files.download("sada_camelbert_finetuned.zip")

## Interpreting results

A high score on this generated corpus is not evidence of real-world success. Discuss the confused categories, dialect gaps, and the need for licensed real data and human label review.